# Microbial Assembly & Annotation using Whole Genome Sequencing (MAC_WGS)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pabloati/MAC_WGS/blob/main/upgrades/MAC_WGS_colab.ipynb)

Welcome to the **MAC_WGS** interactive workshop! In this Google Colab notebook, you will learn the complete computational workflow to filter, assemble, evaluate, circularize, annotate, and benchmark bacterial genomes using Oxford Nanopore Technologies (ONT) long reads and Illumina short reads.

---

### Course Pipeline Workflow

```
┌────────────────────────────────────────────────────────────────────────────────────────┐
│                                 MAC_WGS PIPELINE FLOW                                  │
├────────────────────────────────────────────────────────────────────────────────────────┤
│  1. Raw Read Quality Control (NanoPlot) & Length/Quality Filtering (NanoFilt)         │
│  2. De Novo Long-Read Assembly (Flye --nano-hq)                                       │
│  3. Assembly Graph Inspection (Bandage CLI & Inline Rendering)                        │
│  4. Gene Space Completeness Evaluation (BUSCO - bacillales_odb10)                     │
│  5. Error Correction & Polishing Dynamics (minimap2 + Racon)                           │
│  6. Chromosome Circularization & Reorientation at oriC (dnaapler)                    │
│  7. Prokaryotic Functional Annotation & Genome Mining (Prokka)                        │
│  8. 3-Way Comparative Benchmark on E. coli (Short vs Long vs Hybrid Unicycler)        │
└────────────────────────────────────────────────────────────────────────────────────────┘
```


## ⚙️ Step 0: Environment Setup (Conda / Mamba in Colab)

Because bioinformatics tools require specific C/C++ libraries and pre-compiled binaries, we use `condacolab` to equip this Google Colab instance with a lightweight Mamba package manager.

> [!NOTE]
> Running the cell below will install Mamba. If prompted or if the kernel restarts, simply proceed to the next cell!


In [ ]:
# 1. Install condacolab
!pip install -q condacolab
import condacolab
condacolab.install()


Now, install all required bioinformatics tools (`Flye`, `NanoPlot`, `NanoFilt`, `minimap2`, `Racon`, `dnaapler`, `Prokka`, `BUSCO`, `Bandage`, `Unicycler`, `samtools`) into our environment:


In [ ]:
# 2. Install all bioinformatics tools via Mamba
!mamba install -c conda-forge -c bioconda flye nanoplot nanofilt minimap2 racon dnaapler prokka busco bandage unicycler samtools -y


Let's verify that the tools are successfully installed:


In [ ]:
!flye --version
!prokka --version
!dnaapler --version
!Bandage --version


## 📁 Step 1: Workspace Directory Setup & High-Speed Data Download

In bioinformatics, maintaining an organized workspace is vital. We now download the unified course data archive (`mac_wgs_data.tar.gz`). 

* **High-Speed CDN:** Fetches all *B. subtilis* and *E. coli* sequencing reads directly from the cloud in ~5 seconds.
* **Automatic Layout:** Extracting the archive automatically populates `data/bsubtilis/` and `data/ecoli/`.


In [ ]:
# 1. Download course datasets archive (GitHub Release with automatic ERDA fallback)
!mkdir -p upgrades results
!wget -q --show-progress https://github.com/pabloati/MAC_WGS/releases/download/v1.0-data/mac_wgs_data.tar.gz || \
 (echo "--> Downloading from ERDA backup..." && \
  mkdir -p data/bsubtilis data/ecoli && \
  wget -q -O data/bsubtilis/bsubtilis_long_reads.fastq https://sid.erda.dk/share_redirect/ePr2eWTdSX/data/bsubtilis/bsubtilis_long_reads.fastq && \
  wget -q -O data/ecoli/illumina_f.fq https://sid.erda.dk/share_redirect/ePr2eWTdSX/data/ecoli/illumina_f.fq && \
  wget -q -O data/ecoli/illumina_r.fq https://sid.erda.dk/share_redirect/ePr2eWTdSX/data/ecoli/illumina_r.fq && \
  wget -q -O data/ecoli/minion_2d.fq https://sid.erda.dk/share_redirect/ePr2eWTdSX/data/ecoli/minion_2d.fq)

# 2. Extract archive if downloaded
![ -f mac_wgs_data.tar.gz ] && tar -xzf mac_wgs_data.tar.gz

# 3. Download pre-computed classroom fallback archive
!wget -q -O upgrades/precomputed_results.tar.gz https://raw.githubusercontent.com/pabloati/MAC_WGS/main/upgrades/precomputed_results.tar.gz || true

# 4. Validate dataset presence
!ls -lh data/bsubtilis data/ecoli


---

# Part 1: Long-Read Assembly & Annotation (*Bacillus subtilis*)

---

## 🔬 Step 2: Quality Control & Filtering

### 2.1. Raw Read Quality Assessment with `NanoPlot`
Before running assemblers, we inspect the read length distribution and mean quality scores:


In [ ]:
!mkdir -p results/bsubtilis/qc

!NanoPlot --fastq data/bsubtilis/bsubtilis_long_reads.fastq \
          --outdir results/bsubtilis/qc \
          --prefix bsubtilis_raw_ \
          --threads 2


### Inspecting NanoPlot Summary Statistics:


In [ ]:
!cat results/bsubtilis/qc/bsubtilis_raw_NanoStats.txt


### 2.2. Filtering Low-Quality & Short Reads with `NanoFilt`
Reads shorter than $1,000\text{ bp}$ do not provide sufficient bridging power for repeat resolution and act as graph-tangling noise. We filter for reads $\ge 1,000\text{ bp}$ with a mean Phred score $Q \ge 12$:


In [ ]:
!mkdir -p results/bsubtilis/filtered

!NanoFilt -l 1000 -q 12 data/bsubtilis/bsubtilis_long_reads.fastq \
    > results/bsubtilis/filtered/bsubtilis_filtered.fastq

!ls -lh results/bsubtilis/filtered/bsubtilis_filtered.fastq


## 🧬 Step 3: *De Novo* Long-Read Assembly with `Flye`

We use **Flye**, an assembler based on repeat graphs (ABruijn graph), to assemble the filtered *B. subtilis* reads.

> [!TIP]
> **Memory Optimization:** Running with `--nano-hq` leverages minimizer-based indexing, keeping peak RAM under $1.0\text{ GB}$ and executing in just a few minutes.


In [ ]:
!mkdir -p results/bsubtilis/assembly

!flye --nano-hq results/bsubtilis/filtered/bsubtilis_filtered.fastq \
      --genome-size 4.3m \
      --threads 2 \
      --out-dir results/bsubtilis/assembly


### Inspecting Flye Assembly Information:
Let's check the contig count, lengths, and circularity:


In [ ]:
!cat results/bsubtilis/assembly/assembly_info.txt


## 📊 Step 4: Assembly Graph Visualization with `Bandage`

Assembly graphs (`.gfa` files) reveal the physical topology of the genome. In bacterial isolates, an ideal assembly yields **one big circular fragment (the chromosome ~4.2 Mb)** and potentially smaller independent circular loops (plasmids).

In Colab, we can render the `.gfa` graph into a high-resolution `.png` image and display it directly inline:


In [ ]:
# Render assembly graph to PNG using Bandage CLI
!Bandage image results/bsubtilis/assembly/assembly_graph.gfa results/bsubtilis/assembly/bsubtilis_graph.png --height 600

# Display inline
from IPython.display import Image
Image('results/bsubtilis/assembly/bsubtilis_graph.png')


## 🔍 Step 5: Quality Assessment with `BUSCO`

We assess the completeness of single-copy core genes using the `bacillales_odb10` lineage dataset:


In [ ]:
!mkdir -p results/bsubtilis/busco_flye

!busco -i results/bsubtilis/assembly/assembly.fasta \
       -m genome \
       -l bacillales_odb10 \
       --cpu 2 \
       -o busco_flye \
       --out_path results/bsubtilis/busco_flye


### BUSCO Raw Assembly Summary:


In [ ]:
!cat results/bsubtilis/busco_flye/busco_flye/short_summary.specific.bacillales_odb10.busco_flye.txt


## 🛠️ Step 6: Assembly Polishing with `minimap2` + `Racon`

To correct residual base errors, we map our reads back to the assembly and polish with `Racon`:


In [ ]:
!mkdir -p results/bsubtilis/polished

# 1. Align reads against draft assembly
!minimap2 -x map-ont -t 2 \
          results/bsubtilis/assembly/assembly.fasta \
          results/bsubtilis/filtered/bsubtilis_filtered.fastq \
          > results/bsubtilis/polished/bsubtilis_mapped.paf

# 2. Polish consensus with Racon
!racon -t 2 \
       results/bsubtilis/filtered/bsubtilis_filtered.fastq \
       results/bsubtilis/polished/bsubtilis_mapped.paf \
       results/bsubtilis/assembly/assembly.fasta \
       > results/bsubtilis/polished/bsubtilis_racon.fasta


### Evaluate Polished Assembly with BUSCO:


In [ ]:
!mkdir -p results/bsubtilis/busco_polished

!busco -i results/bsubtilis/polished/bsubtilis_racon.fasta \
       -m genome \
       -l bacillales_odb10 \
       --cpu 2 \
       -o busco_polished \
       --out_path results/bsubtilis/busco_polished


### Compare BUSCO Scores Before vs. After Polishing:


In [ ]:
!cat results/bsubtilis/busco_polished/busco_polished/short_summary.specific.bacillales_odb10.busco_polished.txt


> [!NOTE]
> **Polishing Discussion:** Notice how high-accuracy ONT reads already achieve $\ge 99.8\%$ completeness out-of-the-box. Over-polishing can sometimes slightly alter homopolymer lengths, which is why benchmarking BUSCO before and after polishing is standard best practice!


## 🔄 Step 7: Chromosome Reorientation at *oriC* (`dnaapler`)

By international convention, bacterial chromosomes are oriented so that coordinate 1 begins at the start codon of the **$oriC$ replication initiator gene (`dnaA`)**. This allows cross-assembly synteny comparisons:


In [ ]:
!mkdir -p results/bsubtilis/circularization

!dnaapler all -i results/bsubtilis/assembly/assembly.fasta \
             -o results/bsubtilis/circularization \
             -t 2

# Copy final publication-ready assembly
!cp results/bsubtilis/circularization/dnaapler_reoriented.fasta results/bsubtilis/final_assembly.fasta
!ls -lh results/bsubtilis/final_assembly.fasta


## 🏷️ Step 8: Bacterial Functional Annotation with `Prokka`

`Prokka` coordinates structural gene prediction (`Prodigal` for ORFs, `Aragorn` for tRNAs, `Barrnap` for rRNAs) followed by functional homology searches against curated databases (ISfinder $\rightarrow$ *Bacillus* genus DB $\rightarrow$ UniProt/Swiss-Prot $\rightarrow$ HMMs):


In [ ]:
!mkdir -p results/bsubtilis/annotation

!prokka --outdir results/bsubtilis/annotation \
        --prefix bsubtilis \
        --genus Bacillus \
        --species subtilis \
        --usegenus \
        --cpus 2 \
        results/bsubtilis/final_assembly.fasta


### Inspect Annotation Summary Statistics:


In [ ]:
!cat results/bsubtilis/annotation/bsubtilis.txt


### 💡 Interactive Genome Exploration Challenges

#### Challenge 1: Verify the *dnaA* Replication Synteny Cluster
Let's inspect the first 15 genes in our annotation table (`bsubtilis.tsv`):


In [ ]:
!head -n 15 results/bsubtilis/annotation/bsubtilis.tsv


Notice that **Gene 00001** is `dnaA`, followed by `dnaN`, `recF`, `gyrB`, `gyrA`, and the first complete ribosomal RNA operon ($16\text{S} \rightarrow \text{tRNA-Ile} \rightarrow \text{tRNA-Ala} \rightarrow 23\text{S} \rightarrow 5\text{S}$)!


#### Challenge 2: Calculate the "Dark Matter" Hypothetical Protein Fraction


In [ ]:
total_cds = !grep -c -P "\tCDS\t" results/bsubtilis/annotation/bsubtilis.tsv
hypo_cds = !grep -c "hypothetical protein" results/bsubtilis/annotation/bsubtilis.tsv

t = int(total_cds[0])
h = int(hypo_cds[0])
print(f"Total Predicted CDS: {t}")
print(f"Hypothetical Proteins: {h} ({h/t*100:.2f}%)")


#### Challenge 3: Mine Sporulation Regulators and Surfactin Antibiotic Genes


In [ ]:
# Sporulation master regulators (spo0A, spoIIE)
!grep -E "spo0A|spoIIE" results/bsubtilis/annotation/bsubtilis.tsv

# Surfactin biosynthesis complex
!grep -i "surfactin" results/bsubtilis/annotation/bsubtilis.tsv


---

# Part 2: The 3-Way Comparative Benchmark (*Escherichia coli* C-1)

---

## ⚔️ Step 9: Short Reads (SPAdes) vs. Long Reads (Flye) vs. Hybrid (Unicycler)

In this final module, you will investigate an outbreak isolate of ***E. coli* C-1** by comparing three genomic assembly strategies:

```
┌─────────────────────────────────────────────────────────────────────────────────────────────┐
│                                 THE 3-WAY ASSEMBLY BENCHMARK                                │
├───────────────────────────────┬───────────────────────────────┬─────────────────────────────┤
│   Strategy A: Short-Read      │    Strategy B: Long-Read      │     Strategy C: Hybrid      │
│      (SPAdes / Illumina)      │      (Flye / ONT MinION)      │    (Unicycler: Short+Long)  │
├───────────────────────────────┼───────────────────────────────┼─────────────────────────────┤
│ • Topology: ~40-60 contigs    │ • Topology: 18 contigs (16x)  │ • Topology: 1 circular loop │
│ • Base Accuracy: >99.9% (Q40+)│ • Base Accuracy: Q15-Q20      │ • Base Accuracy: >99.99% (Q50)│
│ • ORFs: Broken by contig ends │ • ORFs: Frameshift splits     │ • ORFs: 100% full-length    │
└───────────────────────────────┴───────────────────────────────┴─────────────────────────────┘
```

### 9.1. Run Long-Read Assembly on *E. coli* with Flye
Let's assemble the *E. coli* MinION dataset:


In [ ]:
!mkdir -p results/ecoli/flye_ecoli

!flye --nano-hq data/ecoli/minion_2d.fq \
      --read-error 0.18 \
      --genome-size 4.6m \
      --threads 2 \
      --out-dir results/ecoli/flye_ecoli


### Inspect Flye Assembly Info on *E. coli*:


In [ ]:
!cat results/ecoli/flye_ecoli/assembly_info.txt


### 9.2. Extract Pre-computed Unicycler Hybrid Assembly Outputs
Running full Unicycler across 8 k-mer sizes requires ~1.5 hours on 2 vCPUs. To save classroom time, we extract the pre-generated Unicycler deliverables:


In [ ]:
# Extract pre-computed deliverables
!tar -xvf upgrades/precomputed_results.tar.gz -C .
!ls -lh results/ecoli/unicycler_ecoli/


### 9.3. Visualizing Repeat Resolution in Bandage (The "Aha!" Moment)

Let's render and compare the **Short-Read Graph** vs. the **Hybrid Bridged Graph** side-by-side:


In [ ]:
# 1. Render SPAdes short-read graph (unresolved repeat knots at rrn operons)
!Bandage image results/ecoli/unicycler_ecoli/001_spades_graph_k127.gfa ecoli_spades_graph.png --height 500

# 2. Render Unicycler hybrid closed circle
!Bandage image results/ecoli/unicycler_ecoli/assembly.gfa ecoli_hybrid_graph.png --height 500

from IPython.display import Image, display

print("=== Strategy A: Short-Read SPAdes Graph (Tangled Knots at 7 rrn Operons) ===")
display(Image('ecoli_spades_graph.png'))

print("=== Strategy C: Unicycler Hybrid Graph (Resolved Single Closed Circle) ===")
display(Image('ecoli_hybrid_graph.png'))


## 📋 Step 10: Benchmark Synthesis & Summary Discussion

| Metric / Dimension | Short Reads Only (SPAdes) | Long Reads Only (Flye) | Hybrid Assembly (Unicycler) | Target Biological Standard |
| :--- | :--- | :--- | :--- | :--- |
| **Input Data Used** | Illumina PE 150 bp | ONT MinION | Illumina PE + ONT MinION | Comprehensive library |
| **Number of Contigs** | ~40 – 70 contigs | **18 contigs** (at 16x depth) | **1 circular contig** | 1 chromosome + plasmids |
| **Contig $N50$** | ~120 – 180 kb | **608 kb** (largest: 748 kb) | **~4.58 Mb** | $pprox 	ext{Total Genome Length}$ |
| **Graph Topology (Bandage)** | Tangled knot (unresolved $rrn$) | Linear subgraphs + repeat nodes | 1 closed circle | Closed loop |
| **Predicted CDS Count** | ~4,200 (broken at contig tips)| ~4,650 (inflated by indels) | **~4,280 (intact ORFs)** | ~4,200–4,400 for *E. coli* |
| **Consensus Base Accuracy**| $> 99.9\%$ ($Q \ge 30$) | ~95–99% ($Q15	ext{--}Q20$) | **$> 99.99\%$ ($Q \ge 50$)** | Publication reference standard |
| **Primary Strength** | Inexpensive, high accuracy | Resolves structural repeats | Gold-standard reference | Complete accuracy & synteny |

---

### Real-World Decision Scenarios:
1. **Hospital Outbreak SNP Tracking (50 isolates over a weekend):** $ightarrow$ **Short-Read (Illumina)** for cost & base accuracy.
2. **Novel 80 kb Pathogenicity Island / AMR Plasmid Discovery:** $ightarrow$ **Long-Read (ONT / PacBio)** to span repeats and confirm circularity.
3. **Official Type Strain GenBank Submission:** $ightarrow$ **Hybrid (or ONT Q20+ Duplex / PacBio HiFi)** for complete structural and sequence fidelity.


## 💾 Step 11: Save Final Results to Google Drive (Optional)

If you would like to keep your generated assemblies and annotations, connect your Google Drive:


In [ ]:
# Optional: Save results to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !tar -czvf /content/drive/MyDrive/MAC_WGS_results.tar.gz results/
print("Workshop completed successfully! All results are stored in results/")
